# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KijoSal-dev/flyrank-ml-internship-wk1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task type: Ranking / scoring. My refresh/content opportunity lane is a ranking problem because the decision is which content items should be reviewed first for a possible refresh. I would assign each content item a priority score and rank the items from highest to lowest. The output would support a content or editorial reviewer who can investigate the highest-priority items first. A wrong ranking could waste reviewer time on a low-value opportunity or cause a stronger opportunity to be missed. The goal is decision-support, not to claim that a refresh will definitely improve performance.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

task_type = "Ranking / scoring"
print("Task type:", task_type)


Task type: Ranking / scoring


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target/proxy: I will use a rule-derived decline_proxy as a proxy for refresh opportunity. The proxy is defined as trend_direction == "down". In the 30,000-item starter dataset, 16,262 items (54.2%) are marked as declining and 13,738 items (45.8%) are not. This proxy is reasonably balanced in this dataset, but it is rule-derived rather than an independently observed future outcome. Therefore, I treat it as a proxy for refresh opportunity rather than a definitive target. A stronger future version of this task would use earlier-period features to predict an independently observed performance outcome in a later time window.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)

# Create a transparent proxy for content decline.
# This is derived from the existing trend_direction rule,
# so it is NOT an independently observed future target.

df["decline_proxy"] = df["trend_direction"].eq("down")

print("Decline proxy counts:")
print(df["decline_proxy"].value_counts(dropna=False))

print("\nDecline proxy proportions:")
print(df["decline_proxy"].value_counts(normalize=True))

df[
    [
        "content_id",
        "trend_direction",
        "trend_pct",
        "decline_proxy"
    ]
].head(10)




Dataset shape: (30000, 44)
Decline proxy counts:
decline_proxy
True     16262
False    13738
Name: count, dtype: int64

Decline proxy proportions:
decline_proxy
True     0.542067
False    0.457933
Name: proportion, dtype: float64


,content_id,trend_direction,trend_pct,decline_proxy
0,content_304f48230142,down,-41.4,True
1,content_a1fb4e703a9e,down,-57.7,True
2,content_9aa793d4d895,down,-60.9,True
3,content_331d6c4de07b,stable,-13.8,False
4,content_d99b7a2d90ca,down,-34.7,True
5,content_d4084a4bc775,down,-38.9,True
6,content_9a34b442b552,down,-92.3,True
7,content_a63219c6e95a,stable,0.6,False
8,content_5e6c160719bc,down,-58.8,True
9,content_c27558df2b0c,down,-29.2,True


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success metric: Precision@50. I will use Precision@50 because the practical action is to give a content reviewer a prioritized queue of potential refresh opportunities. Precision@50 measures the proportion of the top 50 ranked content items that are positive according to the chosen proxy. A higher Precision@50 means that more of the limited review capacity is directed toward items identified as potential opportunities. The current proxy has a positive rate of 54.2%, so this provides a simple reference point for random selection. For example, a Precision@50 of 70% would mean that approximately 35 of the top 50 ranked items are positive, compared with about 27 expected from random selection. I would consider improvement over the baseline directionally useful, while the final operational threshold should be decided after comparing against a simple baseline and considering review capacity. This metric supports decision-making; it does not show that refreshing content will cause performance to improve.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
metric = "Precision@50"
baseline_rate = df["decline_proxy"].mean()

print("Success metric:", metric)
print(f"Proxy positive rate / simple reference baseline: {baseline_rate:.1%}")
print("Example: 70% Precision@50 would mean 35 positive items out of the top 50.")



Success metric: Precision@50
Proxy positive rate / simple reference baseline: 54.2%
Example: 70% Precision@50 would mean 35 positive items out of the top 50.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of analysis: one row represents one pseudonymized content item. The starter dataset contains 30,000 rows and 30,000 unique content_id values, with zero duplicate content IDs, confirming that each row represents a distinct content item. Each item has content, search, traffic, engagement, and trend information. The ranking task would assign a refresh-opportunity score to each content item, producing a prioritized queue with one row per content item. content_id and client_id are identifiers used for grouping and identification, not predictive features.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show the unit of analysis: one row represents one content item.

print("Dataset shape:", df.shape)
print("Unique content IDs:", df["content_id"].nunique())
print("Duplicate content IDs:", df["content_id"].duplicated().sum())

unit_of_analysis = df[
    [
        "content_id",
        "client_id",
        "content_type",
        "word_count",
        "ctr",
        "avg_position",
        "engagement_rate",
        "trend_direction",
        "decline_proxy"
    ]
]

unit_of_analysis.head(10)



Dataset shape: (30000, 45)
Unique content IDs: 30000
Duplicate content IDs: 0


,content_id,client_id,content_type,word_count,ctr,avg_position,engagement_rate,trend_direction,decline_proxy
0,content_304f48230142,client_f369cb89fc,keyword article,3221.0,0.76,10.6,5.88,down,True
1,content_a1fb4e703a9e,client_4e07408562,keyword article,2481.0,0.05,20.3,0.00,down,True
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,3515.0,0.09,36.5,0.00,down,True
3,content_331d6c4de07b,client_19581e27de,keyword article,NaN,0.49,6.2,1.28,stable,False
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,2803.0,0.13,44.0,0.00,down,True
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3080.0,0.03,8.5,0.00,down,True
6,content_9a34b442b552,client_8722616204,keyword article,3059.0,0.00,7.0,0.00,down,True
7,content_a63219c6e95a,client_19581e27de,keyword article,NaN,0.06,21.2,3.57,stable,False
8,content_5e6c160719bc,client_6208ef0f77,keyword article,3807.0,0.09,46.0,5.88,down,True
9,content_c27558df2b0c,client_19581e27de,keyword article,NaN,0.16,4.9,0.00,down,True


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Why ML beats a fixed rule: A simple rule such as trend_direction == "down" can identify content that is already declining, but it does not prioritize which declining items are the most useful refresh opportunities. Content performance depends on multiple signals such as search demand, competition, CTR, average position, engagement, content characteristics, and intent. These signals may interact in ways that are difficult to capture with one fixed threshold or a small set of if-statements. ML is therefore worth testing as a way to combine these signals and produce a ranked decision-support queue. I will not claim that ML is better until it is compared with a simple rule baseline using the chosen metric, Precision@50.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
candidate_signals = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

print("Candidate signals:")
print(candidate_signals)

print("\nNumber of candidate signals:", len(candidate_signals))

print("\nCandidate signal preview:")
df[candidate_signals].head()



Candidate signals:
['search_volume', 'competition', 'cpc', 'word_count', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

Number of candidate signals: 9

Candidate signal preview:


,search_volume,competition,cpc,word_count,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct
0,10.0,0.67,2.05,3221.0,0.76,10.6,5.88,4.55,0.0
1,90.0,0.01,0.05,2481.0,0.05,20.3,0.00,10.00,0.0
2,0.0,0.00,0.00,3515.0,0.09,36.5,0.00,28.57,0.0
3,10.0,0.00,0.00,NaN,0.49,6.2,1.28,3.45,0.0
4,0.0,0.00,0.00,2803.0,0.13,44.0,0.00,24.29,0.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.